In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import importlib
import json
import sys

sys.path.append("")

import pinn_model
pinn_model = importlib.reload(pinn_model)

print(pinn_model.__file__)
print("run_experiment:", hasattr(pinn_model, "run_experiment"))

/home/jupyter/project/pinn_model.py
run_experiment: True


In [2]:
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = "cuda"
    print("gpu:", torch.cuda.get_device_name(0))
else:
    device = "cpu"

work_dir = Path("run")
if not work_dir.exists():
    work_dir = Path(".")

print("device:", device)
print("work_dir:", work_dir)

torch version: 2.0.1+cu118
cuda available: True
gpu: Tesla V100-PCIE-32GB
device: cuda
work_dir: .


In [3]:
m_values = [5, 6, 7, 8, 9, 10, 11, 12]
dtype_values = ["fp32", "fp64"]
seed_values = [0, 1]


base_config = {
    "task_name": "helmholtz1d",
    "dtype": "fp32",
    "seed": 0,
    "device": device,
    "m": 20,
    "lambda_val": 1.0,
    "hid_size": 128,
    "num_layers": 4,
    "n_collocation": 5000,
    "n_bc": 2,
    "adam_steps": 10000,
    "lbfgs_steps": 1000,
    "lr_adam": 5e-4,
    "resample_every": 200,
    "use_adam": True,
    "use_lbfgs": True,
    "lbfgs_tolerance_grad": 1e-12,
    "lbfgs_tolerance_change": 1e-14,
    "lbfgs_history_size": 100,
    "lbfgs_lr": 0.1,
    "lbfgs_line_search_fn": "strong_wolfe",
    "log_dir": str(work_dir / "runs" / "helmholtz_resample_tmp"),
}

len(m_values) * len(dtype_values) * len(seed_values)

32

In [ ]:
all_summaries = []
all_histories = {}

for m in m_values:
    for dtype in dtype_values:
        for seed in seed_values:
            config = base_config.copy()
            config["m"] = m
            config["dtype"] = dtype
            config["seed"] = seed
            config["log_dir"] = str(work_dir / "runs" / f"helmholtz_rs_m{m}_{dtype}_{seed}")

            run_dir = Path(config["log_dir"])
            summary_file = run_dir / "summary.json"
            metrics_file = run_dir / "metrics.csv"

            if summary_file.exists() and metrics_file.exists():
                with open(summary_file) as f:
                    summary = json.load(f)
                history = pd.read_csv(metrics_file)
            else:
                history, summary = pinn_model.run_experiment(config)

            best = history.loc[history["l2_error"].idxmin()]
            summary["m"] = m
            summary["lambda_val"] = config["lambda_val"]
            summary["best_l2_error"] = float(best["l2_error"])
            summary["best_step"] = int(best["step"])
            summary["l2_ratio"] = float(summary["final_l2_error"] / summary["best_l2_error"])
            all_summaries.append(summary)
            all_histories[f"m{m}_{dtype}_{seed}"] = history

            print(m, dtype, seed, summary["final_l2_error"], summary["best_l2_error"], summary["time_sec"])

5 fp32 1 0.00014947904855944216 0.00014947904855944216 180.42691206932068
5 fp64 0 0.0012781916212314252 0.0008974688928242712 154.24301624298096
5 fp64 1 5.777493379241883e-05 5.777493379241883e-05 156.91302680969238
6 fp32 0 8.179871656466275e-05 8.179871656466275e-05 182.94811463356018
6 fp32 1 0.0008209836669266224 0.0004539585206657648 174.20927548408508
6 fp64 0 0.0018195574717574199 0.00025484187680111593 160.21111488342285
6 fp64 1 0.0007587241276229045 0.0007584813423114463 165.78130435943604
7 fp32 0 0.0011489177122712135 0.0011489177122712135 177.93128299713135
7 fp32 1 0.0015475167892873287 0.0015475167892873287 183.59991145133972
7 fp64 0 0.0008842938102300425 0.0006885345669491047 163.7110526561737
7 fp64 1 0.00034036760794367576 0.000340217980358743 160.52083325386047
8 fp32 0 0.0021061119623482227 0.0013765438925474882 180.25551104545593
8 fp32 1 0.0036951485089957714 0.0036951485089957714 181.65801668167114
8 fp64 0 0.0017310368678821544 0.0011055835956741598 159.59601

In [ ]:
df = pd.DataFrame(all_summaries)
df = df.sort_values(["m", "dtype", "seed"]).reset_index(drop=True)

df.to_csv(work_dir / "helmholtz_results_summary.csv", index=False)

cols = ["m", "lambda_val", "dtype", "seed", "final_l2_error", "best_l2_error", "l2_ratio", "best_step", "final_loss", "time_sec", "log_dir"]
df[cols]

In [ ]:
grouped = df.groupby(["m", "dtype"])[["final_l2_error", "best_l2_error", "l2_ratio", "final_loss", "time_sec"]].agg(["mean", "std"])
grouped.to_csv(work_dir / "helmholtz_results_grouped.csv")
grouped

In [ ]:
mean_df = df.groupby(["m", "dtype"])[["final_l2_error", "best_l2_error", "l2_ratio", "final_loss", "time_sec"]].mean().reset_index()

fig, ax = plt.subplots(1, 5, figsize=(22, 4))

for dtype in dtype_values:
    part = mean_df[mean_df["dtype"] == dtype].sort_values("m")
    ax[0].plot(part["m"], part["final_l2_error"], "o-", label=dtype)
    ax[1].plot(part["m"], part["best_l2_error"], "o-", label=dtype)
    ax[2].plot(part["m"], part["l2_ratio"], "o-", label=dtype)
    ax[3].plot(part["m"], part["final_loss"], "o-", label=dtype)
    ax[4].plot(part["m"], part["time_sec"], "o-", label=dtype)

for a in ax:
    a.grid(True)
    a.set_xlabel("m")
    a.legend()

ax[0].set_yscale("log")
ax[1].set_yscale("log")
ax[2].set_yscale("log")
ax[3].set_yscale("log")
ax[0].set_ylabel("final_l2_error")
ax[1].set_ylabel("best_l2_error")
ax[2].set_ylabel("final / best")
ax[3].set_ylabel("final_loss")
ax[4].set_ylabel("time_sec")

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(len(m_values), 2, figsize=(12, 4 * len(m_values)))
ax = np.array(ax).reshape(len(m_values), 2)

for i, m in enumerate(m_values):
    for dtype in dtype_values:
        seed = seed_values[0]
        p = work_dir / "runs" / f"helmholtz_rs_m{m}_{dtype}_{seed}" / "metrics.csv"
        if p.exists():
            h = pd.read_csv(p)
            label = f"{dtype}, seed {seed}"
            ax[i, 0].plot(h["step"], h["l2_error"], label=label)
            ax[i, 1].plot(h["step"], h["total_loss"], label=label)

    ax[i, 0].set_title(f"m={m}")
    ax[i, 1].set_title(f"m={m}")
    ax[i, 0].set_yscale("log")
    ax[i, 1].set_yscale("log")
    ax[i, 0].set_xlabel("step")
    ax[i, 1].set_xlabel("step")
    ax[i, 0].set_ylabel("l2_error")
    ax[i, 1].set_ylabel("total_loss")
    ax[i, 0].grid(True)
    ax[i, 1].grid(True)
    ax[i, 0].legend()
    ax[i, 1].legend()

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(len(m_values), len(dtype_values), figsize=(5 * len(dtype_values), 3 * len(m_values)))
ax = np.array(ax).reshape(len(m_values), len(dtype_values))

for i, m in enumerate(m_values):
    for j, dtype in enumerate(dtype_values):
        seed = seed_values[0]
        p = work_dir / "runs" / f"helmholtz_rs_m{m}_{dtype}_{seed}" / "solution_t1.png"
        ax[i, j].set_title(f"m={m}, {dtype}, seed {seed}")
        ax[i, j].axis("off")
        if p.exists():
            img = plt.imread(p)
            ax[i, j].imshow(img)

fig.tight_layout()
plt.show()